# Homework 3

By Andrew McLaughlin

## Problem 1

The Gold Dragon Coin (GDC) is a unit of currency in Westeros. Let $S$ denote the GDC/USD
exchange rate (the USD value of 1 GDC). Assume $S$ has dynamics

$$
dS_t = (r-q)S_tdt + \sigma(S_t,t)S_tdW_t,
$$

where W is Brownian motion under the [USD] risk-neutral probability measure. The USD interest
rate is $r = 0.06$, the GDC interest rate is $q = 0.01$, today’s time-0 spot is $S_0 = 100$, and

$$
\sigma(S,t) := \mathrm{min}[0.2+5(\mathrm{log}(S/100))^2+0.1\exp{-t}, 0.6].
$$

In [104]:
import numpy as np

### (a)

Find the time-0 price of an American-style put on the GDC. The put has strike 95 and expiry
0.75.

In [105]:
class localvolDynamics:

    def __init__(self, S0, r, q, maxvol, localvol):
        self.S0 = S0
        self.r = r
        self.q = q
        self.maxvol = maxvol
        self.localvol = localvol

In [106]:
hw3dynamics = localvolDynamics(S0 = 100, r = 0.06, q = 0.01, maxvol = 0.6,
                     localvol = lambda S,t: np.minimum(0.2+5*np.log(S/100)**2+0.1*np.exp(-t), 0.6))

# Note that hw3dynamics.localvol is a function
# that may be invoked in the usual way, for example:
# hw2dynamics.localvol( exchangerate , time )

In [107]:
class CallOnAmericanPut:

    def __init__(self, putexpiry, putstrike, callexpiry, callstrike):
        self.putexpiry = putexpiry
        self.putstrike = putstrike
        self.callexpiry = callexpiry
        self.callstrike = callstrike


In [108]:
hw3contract = CallOnAmericanPut(putexpiry=0.75, putstrike=95, callexpiry=0.25, callstrike=10)

In [109]:
class TreeEngine:

    def __init__(self, N):
        self.N = N

    def price_compound_localvol(self, contract, dynamics):
        # Extract parameters
        S0 = dynamics.S0
        r = dynamics.r
        q = dynamics.q
        localvol_func = dynamics.localvol
        T_put = contract.putexpiry  # 0.75
        K_put = contract.putstrike  # 95
        T_call = contract.callexpiry  # 0.25
        K_call = contract.callstrike  # 10
        
        # Compute sigma_avg and tree parameters
        sigma_avg = localvol_func(S0, 0)
        dt = T_put / self.N
        dx = sigma_avg * np.sqrt(3 * dt)
        
        # Check expiry alignment
        if not np.isclose(T_call / dt, round(T_call / dt)):
            raise ValueError("N must be such that call expiry aligns with a tree step (N divisible by 3)")
        M = int(T_call / dt)
        
        # Initialize value arrays
        put_values = [[0.0] * (2 * self.N + 1) for _ in range(self.N + 1)]
        call_values = [[0.0] * (2 * self.N + 1) for _ in range(M + 1)]
        
        # Price the American put (backward induction)
        # At expiry
        for j in range(-self.N, self.N + 1):
            log_S = j * dx
            S = S0 * np.exp(log_S)
            put_values[self.N][j + self.N] = max(K_put - S, 0)
        
        # Backward induction for put
        for k in range(self.N - 1, -1, -1):
            for j in range(-k, k + 1):
                log_S = j * dx
                S = S0 * np.exp(log_S)
                t = k * dt
                sigma = localvol_func(S, t)
                
                # Compute probabilities
                pu_pd = (sigma ** 2 * dt) / (dx ** 2)
                nu = ((r - q) - sigma ** 2 / 2) * dt / dx
                p_u = (pu_pd + nu) / 2
                p_d = (pu_pd - nu) / 2
                p_m = 1 - pu_pd
                
                # Continuation value
                continuation = np.exp(-r * dt) * (
                    p_u * put_values[k + 1][(j + 1) + self.N] +
                    p_m * put_values[k + 1][j + self.N] +
                    p_d * put_values[k + 1][(j - 1) + self.N]
                )
                
                # American exercise
                intrinsic = max(K_put - S, 0)
                put_values[k][j + self.N] = max(intrinsic, continuation)
        
        price_of_put = put_values[0][0 + self.N]
        
        # Price the European call on the put
        # At call expiry
        for j in range(-M, M + 1):
            call_values[M][j + self.N] = max(put_values[M][j + self.N] - K_call, 0)
        
        # Backward induction for call (European, no early exercise)
        for k in range(M - 1, -1, -1):
            for j in range(-k, k + 1):
                log_S = j * dx
                S = S0 * np.exp(log_S)
                t = k * dt
                sigma = localvol_func(S, t)
                
                # Compute probabilities (same as above)
                pu_pd = (sigma ** 2 * dt) / (dx ** 2)
                nu = ((r - q) - sigma ** 2 / 2) * dt / dx
                p_u = (pu_pd + nu) / 2
                p_d = (pu_pd - nu) / 2
                p_m = 1 - pu_pd
                
                # Continuation value
                continuation = np.exp(-r * dt) * (
                    p_u * call_values[k + 1][(j + 1) + self.N] +
                    p_m * call_values[k + 1][j + self.N] +
                    p_d * call_values[k + 1][(j - 1) + self.N]
                )
                
                call_values[k][j + self.N] = continuation
        
        price_of_call_on_put = call_values[0][0 + self.N]
        
        return (price_of_put, price_of_call_on_put)

In [136]:
hw3tree = TreeEngine(N=36)  #change if necessary to get $0.01 accuracy, in your judgment

In [137]:
(answer_part_a, answer_part_b) = hw3tree.price_compound_localvol(hw3contract,hw3dynamics)

In [138]:
answer_part_a

np.float64(7.038543517815775)

- I tested multiple versions of N. Once I increased N above 36, I started seeing instability in the pricing, so I believe 36 provides .01 accuracy of $7.04.

### (b)

Find the time-0 price of a European-style call, with strike 10 and expiry 0.25, on an American
put on the GDC, to be issued at time 0.25 if the European call is exercised (therefore, the
put will not already have been exercised prior to time 0.25). The American put has strike 95
and expiry 0.75.

This call is an example of a compound option. At time 0.25 it gives the call holder the right
to buy the underlying put for 10. The underlying put will have the usual exercise privilege
on the time interval [0.25,0.75], at strike 95.

Example of usage: A company whose expenses are in USD but anticipates receiving a GDC
revenue stream may want to have the put (from part (a)) to hedge the FX risk (specifically,
the risk of GDC weakening against USD). But suppose that the revenue stream may or may
not happen, depending on whether or not the ruling house of Westeros selects this company
as a vendor at time 0.25, so the company does not wish to pay full price for a put that may
or may not turn out to be needed. On the other hand, the company does wish to act soon to
prepare a hedge, because the company fears that GDC will weaken by time 0.25, and the put
will become more expensive. This motivates the company to ask the Iron Bank of Braavos,
what it would cost, to buy a call on the put. Your job is to help the Iron Bank quote a price.
All prices are, as usual, in USD unless stated otherwise.

Complete the coding of the function price compound localvol in the provided ipynb file. Use
a trinomial tree. Your code may reject N for which the call expiry fails to be represented in the tree.
In choosing ∆x, follow L2.33 and choose the “representative” volatility σavg to be σ(S0,0). The
amount of work done by your algorithm in this problem should grow like N2 as N grows (no proof
required). If it grows like N3 in this problem, then your algorithm has some major inefficiency.

In [139]:
answer_part_b

np.float64(1.6082724053386404)

## Problem 2

### (a)

In the Black-Scholes model with interest rate r, no dividends, and volatility σ, approximate
the time-0 delta of an at-the-money (K = S0) vanilla call with expiry T, by applying a
first-order Taylor expansion to the exact formula, and obtaining an explicit approximation
formula in terms of the given parameters.

Then evaluate this approximation to two decimal places, assuming σ = 0.2 and T = 0.25 and
r = 0.01.